# Ungraded Lab 2: LLM Calls and Crafting Simple Augmented Prompts


Welcome to **LLM Calls and Crafting Simple Augmented Prompts**. In this lab, you'll get hands-on practice with using two essential functions that let you interact with Large Language Models (LLMs). These functions help you both send single prompts to an LLM, and have a back-and-forth conversation. The main aim is to show you how to add extra information to your prompts, making them more detailed and useful. This added context helps the model give you better and more precise responses.

In this lab, you'll learn:

- How to set up and send questions to an LLM for both single questions and conversations.
- How to use additional data to make your prompts richer, improving the model's replies.


# Table of Contents
- [ 1 - Understanding the functions to call LLMs](#1)
  - [ 1.1 `generate_with_single_input`](#1-1)
  - [ 1.2 `generate_with_multiple_input`](#1-2)
- [ 2 - Integrating Data into an LLM Prompt](#2)
  - [ 2.1 Understanding the data structure](#2-1)
  - [ 2.2 Creating the Prompt](#2-2)


In [39]:
from utils import (
    generate_with_single_input, 
    generate_with_multiple_input,
    get_proxy_url,
    get_proxy_headers,
    get_together_key
)
from pprint import pprint

In [42]:
import os
from dotenv import load_dotenv, find_dotenv
from pathlib import Path
import httpx

load_dotenv(dotenv_path=Path.cwd() / find_dotenv())

# Verifica se carregou
api_key = os.getenv("OPENAI_API_KEY")
http_client = httpx.Client(verify=False)
if not api_key:
    raise ValueError("OPENAI_API_KEY não foi carregada. Verifique o arquivo .env e a pasta atual.")


### 1.3 Integration with OpenAI library

[Together.ai](together.ai) endpoints are [OpenAI compatible](https://docs.together.ai/docs/openai-api-compatibility) so you can use the [OpenAI library](https://github.com/openai/openai-python) to make the calls. In this section you will explore how to do it.  

In [ ]:
from openai import OpenAI, DefaultHttpxClient
import httpx

In [ ]:
base_url = 'https://api.openai.com/v1' # If using together endpoint, add it here https://api.together.xyz/
llm_model = "gpt-5.4-nano"

# Custom transport to bypass SSL verification. This is only needed if using our proxy. Otherwise you can ignore it.
transport = httpx.HTTPTransport(local_address="0.0.0.0", verify=False)

# Create a DefaultHttpxClient instance with the custom transport
http_client = DefaultHttpxClient(transport=transport, headers=get_proxy_headers())

client = OpenAI(
    api_key = api_key, # Set any as our proxy does not use it. Set the together api key if using the together endpoint.
    base_url=base_url, 
    http_client=http_client, # ssl bypass to make it work via proxy calls, remove it if running with together.ai endpoint 
)

To use it, let's consider the same example as before.

In [ ]:
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

In [ ]:
# Call the ChatCompletion endpoint
response = client.chat.completions.create(
    model=llm_model,
    messages=messages
)

In [ ]:
pprint(response.model_dump())

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': {'annotations': [],
                          'audio': None,
                          'content': 'In 2018, the France team captain was '
                                     '**Hugo Lloris**.',
                          'function_call': None,
                          'refusal': None,
                          'role': 'assistant',
                          'tool_calls': None}}],
 'created': 1784347509,
 'id': 'chatcmpl-E2qaTVad7LV6Blx22WeTZYgJWWERG',
 'model': 'gpt-5.4-nano-2026-03-17',
 'moderation': None,
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 21,
           'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                         'audio_tokens': 0,
                                         'reasoning_tokens': 0,
                                         'r

Notice that the response has several attributes. To access the response content, you may run:response.choices[0].message.content

In [ ]:
pprint(response.choices[0].message.content)

'In 2018, the France team captain was **Hugo Lloris**.'


<a id='2'></a>
## 2 - Integrating Data into an LLM Prompt

In this section, you will learn how to effectively incorporate data into a prompt before passing it to a Large Language Model (LLM). We will work with a small dataset consisting of JSON files that contain information about houses. It will help you understand how to augment prompts in the context of RAG.

<a id='2-1'></a>
### 2.1 Understanding the data structure

Let's have a quick look in the data structure. It is a tiny dataset of houses. A list containing one dictionary per house.

In [45]:
house_data = [
    {
        "address": "123 Maple Street",
        "city": "Springfield",
        "state": "IL",
        "zip": "62701",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Elm Avenue",
        "city": "Shelbyville",
        "state": "TN",
        "zip": "37160",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

<a id='2-2'></a>
### 2.2 Creating the Prompt

Let's begin by constructing the prompt. The first step is to design a layout for the data.

In [46]:
# First, let's create a layout for the houses

def house_info_layout(houses):
    # Create an empty string
    layout = ''
    # Iterate over the houses
    for house in houses:
        # For each house, append the information to the string using f-strings
        # The following way using brackets is a good way to make the code readable as in each line you can start a new f-string that will appended to the previous one
        layout += (f"House located at {house['address']}, {house['city']}, {house['state']} {house['zip']} with "
            f"{house['bedrooms']} bedrooms, {house['bathrooms']} bathrooms, "
            f"{house['square_feet']} sq ft area, priced at ${house['price']}, "
            f"built in {house['year_built']}.\n") # Don't forget the new line character at the end!
    return layout

In [47]:
# Check the layout
print(house_info_layout(house_data))

House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.



Now create a function that generates the prompt to be passed to the Language Learning Model (LLM). The function will take a user-provided query and the available housing data as inputs to effectively address the user's query.

In [48]:
def generate_prompt(query, houses):
    # The code made above is modular enough to accept any list of houses, so you could also choose a subset of the dataset.
    # This might be useful in a more complex context where you want to give only some information to the LLM and not the entire data
    houses_layout = house_info_layout(houses)
    # Create a hard-coded prompt. You can use three double quotes (") in this cases, so you don't need to worry too much about using single or double quotes and breaking the code
    PROMPT = f"""
Use the following houses information to answer users queries.
{houses_layout}
Query: {query}    
             """
    return PROMPT

In [50]:
print(generate_prompt("What is the most expensive house?", houses = house_data))


Use the following houses information to answer users queries.
House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.

Query: What is the most expensive house?    
             


Now let's call the LLM!

In [51]:
query = "What is the most expensive house? And the bigger one?"
messages = [
    {'role': 'user', 'content': query},
]
# Asking without the augmented prompt, let's pass the role as user
query_without_house_info = client.chat.completions.create(model=llm_model,messages=messages)
# With house info, given the prompt structuer, let's pass the role as assistant
enhanced_query = generate_prompt(query, houses = house_data)
messages = [
    {'role': 'assistant', 'content': enhanced_query},
]
query_with_house_info = client.chat.completions.create(model=llm_model,messages=messages)

In [54]:
# Without house info
pprint(query_without_house_info.choices[0].message.content)

('“Most expensive” and “the bigger one” can mean different things (by '
 '**price**, by **size of house**, and/or where—real-world vs. specific '
 'countries). Also, prices change with sales and listings.\n'
 '\n'
 'Here are the commonly cited answers:\n'
 '\n'
 '## 1) Most expensive house (recorded sale)\n'
 '- **Buckingham Palace-sized?** No—this is about *homes* actually sold.\n'
 '- **Most expensive home ever sold (widely cited):** **Antilia** in Mumbai, '
 'India, often described as the world’s most expensive private residence (it’s '
 'a private home, not publicly sold, so “most expensive” is based on estimated '
 'value, not a transaction).\n'
 '  - Estimated value commonly quoted: **~$1–2+ billion** (varies by source).\n'
 '\n'
 '## 2) Bigger one (largest residential home by size)\n'
 '- The “biggest house” most often refers to **floor area**.\n'
 '- **Largest private residence by area (widely cited):** **The Manor (also '
 'called Crystal Palace) in Holmby Hills, Los Angeles**

In [55]:
# With house info
pprint(query_with_house_info.choices[0].message.content)

('- **Most expensive house:** **456 Elm Avenue, Shelbyville, TN 37160** — '
 '**$320,000**  \n'
 '- **Bigger (larger) house:** **456 Elm Avenue, Shelbyville, TN 37160** — '
 '**2,500 sq ft** (vs. 1,500 sq ft)')


Keep it up! You finished the introductory ungraded lab on how to call LLMs and augment prompts!